# 04. Rust evidence appendix: STAMPS parity and native speed

This notebook is the evidence appendix for `start_here_rust.ipynb`.

It answers four concrete questions:
- Is the compiled Rust/native module importable in this checkout?
- Does the focused small-baseline audit artifact show clean MATLAB StaMPS golden-dataset parity?
- Do explicit `pystamps verify` reruns pass for the recorded run/golden pairs?
- Are the measured Rust/native stage 4, 7, and 8 kernels faster than the Python reference while matching numerically?

What this notebook does not claim:
- It does not claim the full four-dataset manifest gate; use `make audit` for that.
- It measures stage-2 topofit Rust exports directly against Python reference outputs.


## 1. Setup and local artifacts

The first output records the Python environment, native module path, Rust source path, and focused audit artifact used by the rest of the notebook.


In [1]:
from __future__ import annotations

import importlib
import importlib.util
import json
import os
import statistics
import subprocess
import sys
import time
from datetime import UTC, datetime
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from pystamps.kernels import (
    describe_backend_matrix,
    run_stage2_grid_accumulate_kernel,
    run_stage2_histogram_kernel,
    run_stage2_topofit_coh_row_invariant_kernel,
    run_stage2_topofit_kernel,
    run_stage2_topofit_row_invariant_kernel,
    run_stage4_edge_stats_kernel,
    run_stage7_scla_kernel,
    run_stage8_edge_noise_kernel,
)


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'pystamps').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from this notebook')


REPO_ROOT = find_repo_root()
FOCUSED_AUDIT_JSON = REPO_ROOT / 'inputs_and_outputs' / 'validation_runs' / 'latest_small_baseline_audit.json'
AUDITED_MANIFEST = REPO_ROOT / 'pystamps' / 'data' / 'audited_workflow_manifest.json'
RUST_SOURCE = REPO_ROOT / 'src' / 'lib.rs'
BASE_ENV = {
    **os.environ,
    'OPENBLAS_NUM_THREADS': '1',
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
    'PYTHONPATH': str(REPO_ROOT),
}


def relpath(path: str | Path | None) -> str:
    if path is None:
        return '<missing>'
    candidate = Path(path)
    try:
        return str(candidate.resolve().relative_to(REPO_ROOT))
    except Exception:
        return str(candidate)


def modified_utc(path: Path | None) -> str:
    if path is None or not path.exists():
        return '<missing>'
    return datetime.fromtimestamp(path.stat().st_mtime, UTC).isoformat(timespec='seconds')


def failure_count(value: object) -> int:
    if isinstance(value, list):
        return len(value)
    if isinstance(value, int):
        return value
    return 0


def run_command(cmd: list[str]) -> tuple[subprocess.CompletedProcess[str], float]:
    t0 = time.perf_counter()
    completed = subprocess.run(
        cmd,
        cwd=REPO_ROOT,
        env=BASE_ENV,
        text=True,
        capture_output=True,
        check=False,
    )
    return completed, time.perf_counter() - t0


native_import_error = None
native_spec = importlib.util.find_spec('pystamps.kernels._stage2_native')
try:
    native_mod = importlib.import_module('pystamps.kernels._stage2_native') if native_spec is not None else None
except Exception as exc:
    native_mod = None
    native_import_error = f'{type(exc).__name__}: {exc}'

native_path = Path(native_spec.origin) if native_spec is not None and native_spec.origin and native_mod is not None else None
artifact_df = pd.DataFrame(
    [
        {'item': 'Python executable', 'value': sys.executable},
        {'item': 'Python version', 'value': sys.version.split()[0]},
        {'item': 'Rust/native module', 'value': relpath(native_path) if native_path else '<not importable>'},
        {'item': 'Rust source', 'value': relpath(RUST_SOURCE)},
        {'item': 'Focused audit artifact', 'value': relpath(FOCUSED_AUDIT_JSON)},
        {'item': 'Audit artifact modified UTC', 'value': modified_utc(FOCUSED_AUDIT_JSON)},
        {'item': 'Native import error', 'value': native_import_error or '<none>'},
    ]
)
display(artifact_df)


,item,value
0,Python executable,/shared/home/rdelprete/PythonProjects/AgenticW...
1,Python version,3.14.2
2,Rust/native module,pystamps/kernels/_stage2_native.cpython-314-x8...
3,Rust source,src/lib.rs
4,Focused audit artifact,inputs_and_outputs/validation_runs/latest_smal...
5,Audit artifact modified UTC,2026-04-22T15:21:37+00:00
6,Native import error,<none>


## 2. Native backend coverage

This table comes from `describe_backend_matrix()`. Only kernels marked `rust_speed_claim = yes` are benchmarked as Rust speed evidence below.


In [2]:
backend_matrix = describe_backend_matrix()
speed_claim_kernels = {
    'stage2_grid_accumulate',
    'stage2_histogram',
    'stage2_topofit',
    'stage2_topofit_coh_row_invariant',
    'stage2_topofit_row_invariant',
    'stage4_edge_stats',
    'stage7_scla',
    'stage8_edge_noise',
}
backend_rows = []
for kernel, info in sorted(backend_matrix.get('kernels', {}).items()):
    supported = info.get('supported_backends', [])
    available = info.get('available_backends', [])
    backend_rows.append(
        {
            'kernel': kernel,
            'python_reference': info.get('baseline_backend', 'python'),
            'native_available_now': 'native' in available,
            'available_backends': ', '.join(available),
            'rust_speed_claim': 'yes' if kernel in speed_claim_kernels and 'native' in available else 'no',
            'reader_note': (
                'measured below against Python'
                if kernel in speed_claim_kernels and 'native' in available
                else 'native backend is not available in this environment'
            ),
        }
    )
backend_df = pd.DataFrame(backend_rows)
display(backend_df)

,kernel,python_reference,native_available_now,available_backends,rust_speed_claim,reader_note
0,stage2_grid_accumulate,python,True,"native, python",yes,measured below against Python
1,stage2_histogram,python,True,"native, python",yes,measured below against Python
2,stage2_topofit,python,True,"native, python",yes,measured below against Python
3,stage2_topofit_coh_row_invariant,python,True,"native, python",yes,measured below against Python
4,stage2_topofit_row_invariant,python,True,"native, python",yes,measured below against Python
5,stage4_edge_stats,python,True,"native, python",yes,measured below against Python
6,stage7_scla,python,True,"cuda, native, python",yes,measured below against Python
7,stage8_edge_noise,python,True,"cuda, native, python",yes,measured below against Python


## 3. Focused StaMPS golden-dataset parity artifact

The audit artifact compares generated pySTAMPS outputs with recorded MATLAB StaMPS golden roots. The useful read is simple: completed must be true, interrupted must be false, and every audit row must be ok.


In [3]:
if not FOCUSED_AUDIT_JSON.exists():
    raise FileNotFoundError(f'Focused audit artifact is missing: {FOCUSED_AUDIT_JSON}')

audit_payload = json.loads(FOCUSED_AUDIT_JSON.read_text(encoding='utf-8'))
audit_rows = audit_payload.get('audits', [])
audit_ok = bool(audit_payload.get('ok') and audit_payload.get('completed') and not audit_payload.get('interrupted'))

audit_status = pd.DataFrame(
    [
        {'check': 'artifact exists', 'result': FOCUSED_AUDIT_JSON.exists()},
        {'check': 'audit completed', 'result': bool(audit_payload.get('completed'))},
        {'check': 'audit interrupted', 'result': bool(audit_payload.get('interrupted'))},
        {'check': 'overall focused audit ok', 'result': audit_ok},
        {'check': 'full manifest claimed here', 'result': False},
    ]
)

audit_df = pd.DataFrame(
    [
        {
            'dataset': audit.get('dataset'),
            'workflow': audit.get('workflow'),
            'status': audit.get('status'),
            'ok': bool(audit.get('ok')),
            'checked_files': int(audit.get('checked', 0)),
            'failed_files': failure_count(audit.get('failed', [])),
            'run_root': relpath(audit.get('run_root')),
            'golden_root': relpath(audit.get('golden_root')),
        }
        for audit in audit_rows
    ]
)

display(audit_status)
display(audit_df)


,check,result
0,artifact exists,True
1,audit completed,True
2,audit interrupted,False
3,overall focused audit ok,True
4,full manifest claimed here,False


,dataset,workflow,status,ok,checked_files,failed_files,run_root,golden_root
0,InSAR_dataset_small_baseline_stage7diag,InSAR_dataset_small_baseline_stage7diag_audit,passed,True,4,0,inputs_and_outputs/validation_runs/20260422_15...,inputs_and_outputs/InSAR_dataset_small_baselin...
1,InSAR_dataset_small_baseline_stage7,InSAR_dataset_small_baseline_stage7_audit,passed,True,4,0,inputs_and_outputs/validation_runs/20260422_15...,inputs_and_outputs/InSAR_dataset_small_baselin...


## 4. Step-by-step parity evaluation

This is the parity logic in human terms. The next cell states each step and the evidence backing it.


In [4]:
parity_steps = pd.DataFrame(
    [
        {
            'step': '1. Pick a trusted answer',
            'what_to_check': 'The golden roots are MATLAB StaMPS reference outputs.',
            'evidence': '; '.join(sorted(relpath(audit.get('golden_root')) for audit in audit_rows)),
            'result': 'PASS' if audit_rows else 'FAIL',
        },
        {
            'step': '2. Regenerate pySTAMPS output',
            'what_to_check': 'The audit rows point at fresh validation run roots.',
            'evidence': '; '.join(sorted(relpath(audit.get('run_root')) for audit in audit_rows)),
            'result': 'PASS' if audit_rows else 'FAIL',
        },
        {
            'step': '3. Compare files',
            'what_to_check': 'Every recorded audit row passed with zero failed files.',
            'evidence': f"{int(audit_df['checked_files'].sum()) if not audit_df.empty else 0} checked, {int(audit_df['failed_files'].sum()) if not audit_df.empty else 0} failed",
            'result': 'PASS' if audit_ok else 'FAIL',
        },
        {
            'step': '4. State the limit',
            'what_to_check': 'This notebook proves focused small-baseline stage-7 parity, not the full manifest gate.',
            'evidence': 'Use make audit for full manifest parity.',
            'result': 'PASS',
        },
    ]
)
display(parity_steps)


,step,what_to_check,evidence,result
0,1. Pick a trusted answer,The golden roots are MATLAB StaMPS reference o...,inputs_and_outputs/InSAR_dataset_small_baselin...,PASS
1,2. Regenerate pySTAMPS output,The audit rows point at fresh validation run r...,inputs_and_outputs/validation_runs/20260422_15...,PASS
2,3. Compare files,Every recorded audit row passed with zero fail...,"8 checked, 0 failed",PASS
3,4. State the limit,This notebook proves focused small-baseline st...,Use make audit for full manifest parity.,PASS


## 5. Explicit `pystamps verify` reruns

The audit artifact is useful, but this notebook also reruns `pystamps verify` for each recorded run root against its MATLAB StaMPS golden root.


In [5]:
verify_rows: list[dict[str, object]] = []
for audit in audit_rows:
    cmd = [
        sys.executable,
        '-m',
        'pystamps.cli',
        'verify',
        '--run',
        str(audit.get('run_root')),
        '--golden',
        str(audit.get('golden_root')),
    ]
    completed, elapsed = run_command(cmd)
    try:
        payload = json.loads(completed.stdout)
    except json.JSONDecodeError:
        payload = {'ok': False, 'checked': 0, 'failed': ['stdout was not JSON']}
    verify_rows.append(
        {
            'dataset': audit.get('dataset'),
            'returncode': completed.returncode,
            'elapsed_sec': round(elapsed, 3),
            'ok': bool(payload.get('ok')),
            'checked_files': int(payload.get('checked', 0)),
            'failed_files': failure_count(payload.get('failed', [])),
            'run_root': relpath(audit.get('run_root')),
            'golden_root': relpath(audit.get('golden_root')),
        }
    )

verify_df = pd.DataFrame(verify_rows)
verify_ok = bool(not verify_df.empty and verify_df['ok'].all() and (verify_df['returncode'] == 0).all())
display(verify_df)


,dataset,returncode,elapsed_sec,ok,checked_files,failed_files,run_root,golden_root
0,InSAR_dataset_small_baseline_stage7diag,0,12.916,True,4,0,inputs_and_outputs/validation_runs/20260422_15...,inputs_and_outputs/InSAR_dataset_small_baselin...
1,InSAR_dataset_small_baseline_stage7,0,10.581,True,4,0,inputs_and_outputs/validation_runs/20260422_15...,inputs_and_outputs/InSAR_dataset_small_baselin...


## 6. Native kernel smoke tests

This is a focused pytest selection for native-vs-Python numerical parity. It is intentionally much smaller than `make audit`.


In [6]:
pytest_expr = (
    'stage4_stage7_stage8_native_kernels_match_python_reference '
    'or stage2_histogram_kernel_native_equal_spacing_matches_cpu'
)
if native_mod is None:
    pytest_completed = None
    pytest_ok = False
    pytest_df = pd.DataFrame(
        [
            {
                'selection': pytest_expr,
                'returncode': '<skipped>',
                'ok': False,
                'reason': 'native extension is not importable in this environment',
            }
        ]
    )
else:
    pytest_cmd = [sys.executable, '-m', 'pytest', '-q', 'tests/test_kernels_accelerated.py', '-k', pytest_expr]
    pytest_completed, pytest_elapsed = run_command(pytest_cmd)
    pytest_ok = pytest_completed.returncode == 0
    pytest_df = pd.DataFrame(
        [
            {
                'selection': pytest_expr,
                'returncode': pytest_completed.returncode,
                'elapsed_sec': round(pytest_elapsed, 3),
                'ok': pytest_ok,
                'stdout_tail': '\n'.join(pytest_completed.stdout.strip().splitlines()[-3:]),
                'stderr_tail': '\n'.join(pytest_completed.stderr.strip().splitlines()[-3:]),
            }
        ]
    )
display(pytest_df)


,selection,returncode,elapsed_sec,ok,stdout_tail,stderr_tail
0,stage4_stage7_stage8_native_kernels_match_pyth...,0,14.991,True,.. ...,


## 7. Rust/native speed benchmark

This benchmark compares Python and compiled Rust/native kernels for every native kernel listed above: the stage 2 kernels plus stage 4 edge statistics, stage 7 SCLA, and stage 8 edge noise.


In [7]:
bench_rows: list[dict[str, object]] = []


def timed(fn: Callable[[], Any], repeats: int) -> tuple[list[float], Any]:
    fn()
    runs: list[float] = []
    output: Any = None
    for _ in range(repeats):
        t0 = time.perf_counter()
        output = fn()
        runs.append(time.perf_counter() - t0)
    return runs, output


def output_items(value: Any) -> dict[str, np.ndarray]:
    if isinstance(value, dict):
        return {str(key): np.asarray(item) for key, item in value.items()}
    if isinstance(value, tuple):
        return {f'out_{idx}': np.asarray(item) for idx, item in enumerate(value)}
    return {'out': np.asarray(value)}


def max_abs_array(left: np.ndarray, right: np.ndarray) -> float:
    left_arr = np.asarray(left)
    right_arr = np.asarray(right)
    if left_arr.shape != right_arr.shape:
        return float('inf')
    if np.array_equal(left_arr, right_arr, equal_nan=True):
        return 0.0
    finite = np.isfinite(left_arr) & np.isfinite(right_arr)
    if not np.array_equal(left_arr[~finite], right_arr[~finite], equal_nan=True):
        return float('inf')
    return float(np.max(np.abs(left_arr[finite] - right_arr[finite]))) if np.any(finite) else 0.0


def max_abs_output(left: Any, right: Any) -> float:
    left_items = output_items(left)
    right_items = output_items(right)
    if set(left_items) != set(right_items):
        return float('inf')
    return max(max_abs_array(left_items[key], right_items[key]) for key in sorted(left_items))


def native_topofit_generic(cpxphase: np.ndarray, bperp: np.ndarray, n_trial_wraps: float, threads: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if native_mod is None:
        raise RuntimeError('native extension is not importable')
    result = native_mod.ps_topofit_batch_generic(
        np.ascontiguousarray(cpxphase, dtype=np.complex128),
        np.ascontiguousarray(bperp, dtype=np.float64),
        float(n_trial_wraps),
        int(threads),
    )
    return tuple(np.asarray(item) for item in result)


def native_topofit_row(cpxphase: np.ndarray, bperp_vec: np.ndarray, n_trial_wraps: float, threads: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if native_mod is None:
        raise RuntimeError('native extension is not importable')
    result = native_mod.ps_topofit_batch_row_invariant(
        np.ascontiguousarray(cpxphase, dtype=np.complex128),
        np.ascontiguousarray(bperp_vec, dtype=np.float64),
        float(n_trial_wraps),
        int(threads),
    )
    return tuple(np.asarray(item) for item in result)


def native_topofit_row_coh(cpxphase: np.ndarray, bperp_vec: np.ndarray, n_trial_wraps: float, threads: int) -> np.ndarray:
    if native_mod is None:
        raise RuntimeError('native extension is not importable')
    return np.asarray(
        native_mod.ps_topofit_coh_row_invariant(
            np.ascontiguousarray(cpxphase, dtype=np.complex128),
            np.ascontiguousarray(bperp_vec, dtype=np.float64),
            float(n_trial_wraps),
            int(threads),
        ),
        dtype=np.float64,
    )


def record_kernel(
    kernel: str,
    python_fn: Callable[[], Any],
    native_fn: Callable[[], Any],
    tolerance: float,
    repeats: int,
) -> None:
    python_runs, python_output = timed(python_fn, repeats)
    native_runs, native_output = timed(native_fn, repeats)
    python_mean = statistics.mean(python_runs)
    native_mean = statistics.mean(native_runs)
    max_abs = max_abs_output(python_output, native_output)
    bench_rows.append(
        {
            'kernel': kernel,
            'python_mean_sec': python_mean,
            'native_mean_sec': native_mean,
            'speedup_vs_python': python_mean / native_mean if native_mean > 0 else float('inf'),
            'max_abs': max_abs,
            'tolerance': tolerance,
            'parity_ok': bool(max_abs <= tolerance),
            'rust_faster': bool(native_mean < python_mean),
            'python_runs_sec': ', '.join(f'{value:.4f}' for value in python_runs),
            'native_runs_sec': ', '.join(f'{value:.4f}' for value in native_runs),
        }
    )


if native_mod is None:
    bench_rows.append(
        {
            'kernel': '<native unavailable>',
            'parity_ok': False,
            'rust_faster': False,
            'reason': 'build/install the Rust extension before measuring native speed',
        }
    )
else:
    repeats = int(os.environ.get('PYSTAMPS_NOTEBOOK_BENCH_REPEATS', '2'))
    native_threads = int(os.environ.get('PYSTAMPS_NOTEBOOK_NATIVE_THREADS', '8'))

    rng_grid = np.random.default_rng(101)
    n_grid_ps = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_GRID_PS', '100000'))
    n_grid_ifg = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_GRID_IFG', '24'))
    n_grid_i = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_GRID_I', '500'))
    n_grid_j = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_GRID_J', '20'))
    ph_weight = np.exp(1j * rng_grid.normal(size=(n_grid_ps, n_grid_ifg))).astype(np.complex64)
    grid_lin = rng_grid.integers(0, n_grid_i * n_grid_j, size=n_grid_ps, dtype=np.int64)
    record_kernel(
        'stage2_grid_accumulate',
        lambda: run_stage2_grid_accumulate_kernel(ph_weight, grid_lin, n_grid_i, n_grid_j, backend='python'),
        lambda: run_stage2_grid_accumulate_kernel(ph_weight, grid_lin, n_grid_i, n_grid_j, backend='native', threads=native_threads),
        1.0e-6,
        repeats,
    )

    rng_hist = np.random.default_rng(102)
    hist_values = rng_hist.normal(size=int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_HIST_VALUES', '2000000'))).astype(np.float64)
    hist_centers = np.linspace(-5.0, 5.0, int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_HIST_CENTERS', '401')), dtype=np.float64)
    record_kernel(
        'stage2_histogram',
        lambda: run_stage2_histogram_kernel(hist_values, hist_centers, backend='python'),
        lambda: run_stage2_histogram_kernel(hist_values, hist_centers, backend='native'),
        0.0,
        repeats,
    )

    rng_topo = np.random.default_rng(103)
    n_topo_ps = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_TOPOFIT_PS', '3000'))
    n_topo_ifg = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_TOPOFIT_IFG', '28'))
    cpx_topo = np.exp(1j * rng_topo.normal(size=(n_topo_ps, n_topo_ifg))).astype(np.complex128)
    bperp_topo = rng_topo.normal(size=(n_topo_ps, n_topo_ifg)).astype(np.float64)
    record_kernel(
        'stage2_topofit',
        lambda: run_stage2_topofit_kernel(cpx_topo, bperp_topo, 1.5, backend='python'),
        lambda: native_topofit_generic(cpx_topo, bperp_topo, 1.5, native_threads),
        1.0e-9,
        repeats,
    )

    rng_topo_row = np.random.default_rng(99)
    cpx_row = np.exp(1j * rng_topo_row.normal(size=(n_topo_ps, n_topo_ifg))).astype(np.complex128)
    bperp_row = np.linspace(-120.0, 120.0, n_topo_ifg, dtype=np.float64)
    bperp_row_mat = np.broadcast_to(bperp_row, (n_topo_ps, n_topo_ifg)).copy()
    record_kernel(
        'stage2_topofit_row_invariant',
        lambda: run_stage2_topofit_row_invariant_kernel(cpx_row, bperp_row_mat, 1.5, backend='python'),
        lambda: native_topofit_row(cpx_row, bperp_row, 1.5, native_threads),
        1.0e-9,
        repeats,
    )

    rng_topo_coh = np.random.default_rng(104)
    n_coh_ps = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_COH_PS', '5000'))
    n_coh_ifg = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE2_COH_IFG', '80'))
    cpx_coh = np.exp(1j * rng_topo_coh.normal(size=(n_coh_ps, n_coh_ifg))).astype(np.complex128)
    bperp_coh = np.linspace(-120.0, 120.0, n_coh_ifg, dtype=np.float64)
    bperp_coh_mat = np.broadcast_to(bperp_coh, (n_coh_ps, n_coh_ifg)).copy()
    record_kernel(
        'stage2_topofit_coh_row_invariant',
        lambda: run_stage2_topofit_coh_row_invariant_kernel(cpx_coh, bperp_coh_mat, 1.5, backend='python'),
        lambda: native_topofit_row_coh(cpx_coh, bperp_coh, 1.5, native_threads),
        1.0e-9,
        repeats,
    )

    rng4 = np.random.default_rng(321)
    n_ps4 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE4_PS', '30000'))
    n_ifg4 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE4_IFG', '32'))
    n_edge4 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE4_EDGES', '60000'))
    ph_weed = np.exp(1j * rng4.normal(size=(n_ps4, n_ifg4))).astype(np.complex128)
    node_a4 = rng4.integers(0, n_ps4, size=n_edge4, dtype=np.int64)
    node_b4 = rng4.integers(0, n_ps4, size=n_edge4, dtype=np.int64)
    bperp4 = np.linspace(-80.0, 120.0, n_ifg4, dtype=np.float64)
    day4 = np.linspace(0.0, 365.0, n_ifg4, dtype=np.float64)
    record_kernel(
        'stage4_edge_stats',
        lambda: run_stage4_edge_stats_kernel(ph_weed, node_a4, node_b4, bperp4, day4, 360.0, False, backend='python'),
        lambda: run_stage4_edge_stats_kernel(ph_weed, node_a4, node_b4, bperp4, day4, 360.0, False, backend='native'),
        1.0e-10,
        repeats,
    )

    rng7 = np.random.default_rng(322)
    n_ps7 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE7_PS', '30000'))
    n_ifg7 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE7_IFG', '36'))
    ph_proc = rng7.normal(size=(n_ps7, n_ifg7)).astype(np.float64)
    ph_mean_v = rng7.normal(size=(n_ps7, n_ifg7)).astype(np.float64)
    bperp_mat = rng7.normal(size=(n_ps7, n_ifg7)).astype(np.float64)
    unwrap_ix = np.arange(n_ifg7, dtype=np.int64)
    solve_ix = np.arange(n_ifg7, dtype=np.int64)
    day7 = np.linspace(0.0, 365.0, n_ifg7, dtype=np.float64)
    ifg_std = np.full(n_ifg7, 10.0, dtype=np.float64)
    record_kernel(
        'stage7_scla',
        lambda: run_stage7_scla_kernel(ph_proc, ph_mean_v, bperp_mat, unwrap_ix, solve_ix, day7, 1, ifg_std, backend='python'),
        lambda: run_stage7_scla_kernel(ph_proc, ph_mean_v, bperp_mat, unwrap_ix, solve_ix, day7, 1, ifg_std, backend='native'),
        1.0e-4,
        repeats,
    )

    rng8 = np.random.default_rng(323)
    n_ps8 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE8_PS', '50000'))
    n_ifg8 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE8_IFG', '36'))
    n_edge8 = int(os.environ.get('PYSTAMPS_NOTEBOOK_STAGE8_EDGES', '120000'))
    uw_ph = np.exp(1j * rng8.normal(size=(n_ps8, n_ifg8))).astype(np.complex64)
    node_a8 = rng8.integers(0, n_ps8, size=n_edge8, dtype=np.int64)
    node_b8 = rng8.integers(0, n_ps8, size=n_edge8, dtype=np.int64)
    record_kernel(
        'stage8_edge_noise',
        lambda: run_stage8_edge_noise_kernel(uw_ph, node_a8, node_b8, backend='python'),
        lambda: run_stage8_edge_noise_kernel(uw_ph, node_a8, node_b8, backend='native'),
        1.0e-5,
        repeats,
    )

bench_df = pd.DataFrame(bench_rows)
if not bench_df.empty and {'speedup_vs_python', 'parity_ok', 'rust_faster'} <= set(bench_df.columns):
    speed_ok = bool(bench_df['parity_ok'].all() and bench_df['rust_faster'].all())
    min_speedup = round(float(bench_df['speedup_vs_python'].min()), 3)
else:
    speed_ok = False
    min_speedup = '<not measured>'

bench_display = bench_df.copy()
for column in ['python_mean_sec', 'native_mean_sec', 'speedup_vs_python', 'max_abs', 'tolerance']:
    if column in bench_display:
        bench_display[column] = bench_display[column].map(lambda value: round(float(value), 6) if pd.notna(value) else value)

speed_summary = pd.DataFrame(
    [
        {'check': 'all measured Rust/native kernels match Python numerically', 'result': bool(bench_df.get('parity_ok', pd.Series(dtype=bool)).all())},
        {'check': 'all measured Rust/native kernels are faster', 'result': bool(bench_df.get('rust_faster', pd.Series(dtype=bool)).all())},
        {'check': 'benchmarked Rust/native kernels', 'result': ', '.join(bench_df['kernel'].astype(str)) if not bench_df.empty else '<none>'},
        {'check': 'minimum native speedup versus Python', 'result': min_speedup},
    ]
)

display(bench_display.reindex(columns=['kernel', 'python_mean_sec', 'native_mean_sec', 'speedup_vs_python', 'max_abs', 'tolerance', 'parity_ok', 'rust_faster', 'python_runs_sec', 'native_runs_sec', 'reason']).dropna(axis=1, how='all'))
display(speed_summary)

,kernel,python_mean_sec,native_mean_sec,speedup_vs_python,max_abs,tolerance,parity_ok,rust_faster,python_runs_sec,native_runs_sec
0,stage2_grid_accumulate,0.027620,0.006688,4.129618,0.000000,0.000001,True,True,"0.0265, 0.0288","0.0059, 0.0075"
1,stage2_histogram,0.150056,0.108788,1.379346,0.000000,0.000000,True,True,"0.1516, 0.1485","0.1081, 0.1094"
2,stage2_topofit,0.463211,0.010476,44.218212,0.000000,0.000000,True,True,"0.4650, 0.4614","0.0097, 0.0113"
3,stage2_topofit_row_invariant,0.459299,0.003770,121.837922,0.000000,0.000000,True,True,"0.4596, 0.4590","0.0037, 0.0038"
4,stage2_topofit_coh_row_invariant,0.053620,0.010435,5.138634,0.000000,0.000000,True,True,"0.0534, 0.0538","0.0106, 0.0103"
5,stage4_edge_stats,5.177739,0.855843,6.049871,0.000000,0.000000,True,True,"5.2315, 5.1240","0.8393, 0.8724"
6,stage7_scla,0.078962,0.014766,5.347708,0.000031,0.000100,True,True,"0.0766, 0.0813","0.0146, 0.0149"
7,stage8_edge_noise,0.101659,0.034459,2.950124,0.000000,0.000010,True,True,"0.1072, 0.0961","0.0343, 0.0347"


,check,result
0,all measured Rust/native kernels match Python ...,True
1,all measured Rust/native kernels are faster,True
2,benchmarked Rust/native kernels,"stage2_grid_accumulate, stage2_histogram, stag..."
3,minimum native speedup versus Python,1.379


## 8. Final verdict

The final table is the compact answer. Every evidence check should pass, while the two non-claims should remain false.


In [8]:
final_verdict = pd.DataFrame(
    [
        {'claim': 'Rust/native module is importable', 'ok': native_mod is not None},
        {'claim': 'Focused audit completed cleanly', 'ok': audit_ok},
        {'claim': 'Explicit verify reruns passed', 'ok': verify_ok},
        {'claim': 'Focused native kernel pytest passed', 'ok': pytest_ok},
        {'claim': 'Every measured Rust/native kernel matches Python and is faster', 'ok': speed_ok},
        {'claim': 'Full manifest parity claimed in this notebook', 'ok': False},
    ]
)
display(final_verdict)

,claim,ok
0,Rust/native module is importable,True
1,Focused audit completed cleanly,True
2,Explicit verify reruns passed,True
3,Focused native kernel pytest passed,True
4,Every measured Rust/native kernel matches Pyth...,True
5,Full manifest parity claimed in this notebook,False


## How to use this result

If the first five final-verdict rows are true, this notebook supports the focused Rust/native claim: small-baseline stage-7 parity against recorded MATLAB StaMPS golden datasets, plus measured speedups for the Rust-backed stage 4, 7, and 8 kernels.

For the full maintained parity gate, run `make audit` and use the manifest-driven validation report.
